# Autograd: Automatic Differentiation Engine

Reach for this when you need: 
- To understand how `requires_grad` and `backward()` work.
- Reference for non-differentiable operations (e.g., slicing, indexing).
- Implementation of custom autograd functions (advanced research).

In [1]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 1. Tracking Gradients

**requires_grad**
- `requires_grad=True`: PyTorch will track all operations on this tensor for backprop.
- `grad_fn`: Stores the function that created the tensor (e.g., `AddBackward0`, `MulBackward0`).

✅ **Use when**: Initializing trainable parameters.
❌ **Don't use when**: Constant weights or inputs that don't need optimization (saves memory).

In [11]:
x = torch.tensor([1., 2.], requires_grad=True)
y = x + 2 # y has a grad_fn
z = y * y * 3
out = z.mean()

# Compute gradients
out.backward()

# Access gradients: x.grad stores d(out)/dx
print(f"Gradient of x: {x.grad}")

Gradient of x: tensor([ 9., 12.])


## 2. Gradient Control: Context Managers

### `torch.no_grad()`
Disables gradient tracking permanently for the block.

✅ **Use when**: Inference, validation loops, or manual parameter updates.
❌ **Don't use when**: You need to calculate gradients for training.

In [9]:
with torch.no_grad():
    y = x * 2
    print(f"y.requires_grad: {y.requires_grad}") # result: False

y.requires_grad: False


### `detach()`
Creates a new tensor that shares the same storage but does not require gradients.

✅ **Use when**: Part of a network should not be updated (e.g., freezing a backbone).
❌ **Don't use when**: You want to permanently disable gradients for an entire loop.

In [13]:
z = x.detach()
print(f"z.requires_grad: {z.requires_grad}") # result: False

z.requires_grad: False


## 3. Custom Autograd Functions (Advanced)

| Method | Description |
| :--- | :--- |
| `forward` | Perform the operation on the input tensor |
| `backward` | Calculate and return the gradient w.r.t the input |

✅ **Use when**: Implementing a custom layer with a known non-standard gradient.
❌ **Don't use when**: The function can already be expressed using standard PyTorch ops.

In [14]:
class MyReLU(torch.autograd.Function):
    @staticmethod
    def forward(ctx, input_tensor):
        # Save for backward pass
        ctx.save_for_backward(input_tensor)
        return input_tensor.clamp(min=0)

    @staticmethod
    def backward(ctx, grad_output):
        input_tensor, = ctx.saved_tensors
        grad_input = grad_output.clone()
        grad_input[input_tensor < 0] = 0
        return grad_input

# Usage
relu = MyReLU.apply
out = relu(torch.randn(3, requires_grad=True))

### Common Pitfalls
- **In-place updates**: `x += 1` on a tensor requiring grad will often fail. Use `x = x + 1` or perform updates inside `with torch.no_grad():` block.
- **Accumulated Gradients**: `backward()` accumulates (adds) gradients. ALWAYS call `optimizer.zero_grad()` before the backward pass in training.
- **CPU vs GPU**: Gradients follow the tensor device. Ensure targets and weights are on the same device before `backward()`.

### Key Takeaways
- `requires_grad=True` is the entry point for optimization.
- Use `detach()` to stop gradients from flowing through a specific branch in the graph.
- Custom functions require both `forward` and `backward` methods defined as `@staticmethod`.